# Analisis Statistik Label dan Pemotongan Citra (Tiling) untuk Skripsi BAB 4.2

Notebook ini digunakan untuk menghitung statistika deskriptif target label UNOSAT dan sebaran data tile setelah pemotongan (*tiling*) untuk wilayah studi di **BAB 4.2 (Pembuatan Label Target dan Tiling)**.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from osgeo import gdal

ROOT = Path("..").resolve()
PREPROCESSED_ROOT = ROOT / "dataset/features_preprocessed"
LABEL_ROOT = ROOT / "dataset/labels_unosat_rasterized"

print(f"ROOT Workspace: {ROOT}")

ROOT Workspace: /home/nozomi/Productive/skripsi


## 1. Statistik Deskriptif Hasil Rasterisasi Label Target per Wilayah

Selanjutnya kita akan menghitung jumlah piksel valid batas, piksel valid label, piksel banjir UNOSAT, dan piksel air/sungai permanen untuk masing-masing wilayah kajian pada citra raster utuh (sebelum tiling).

In [2]:
regions = [
    'Aceh_Besar', 'Aceh_Tamiang', 'Aceh_Timur', 'Aceh_Utara', 'Agam',
    'Banda_Aceh', 'Bireuen', 'Langsa', 'Pasaman_Barat', 'Pidie', 'Pidie_Jaya'
]

rows = []
for r in regions:
    r_dir = PREPROCESSED_ROOT / r
    
    # Read boundary mask
    ds_feat = gdal.Open(str(r_dir / "feature_valid_mask.tif"))
    feat_valid = ds_feat.GetRasterBand(1).ReadAsArray().astype(bool)
    ds_feat = None
    
    # Read label masks
    ds_flood = gdal.Open(str(LABEL_ROOT / r / "label_flood_binary.tif"))
    flood = ds_flood.GetRasterBand(1).ReadAsArray().astype(bool)
    ds_flood = None
    
    ds_water = gdal.Open(str(LABEL_ROOT / r / "label_water_river_mask.tif"))
    water = ds_water.GetRasterBand(1).ReadAsArray().astype(bool)
    ds_water = None
    
    ds_label = gdal.Open(str(LABEL_ROOT / r / "label_valid_mask.tif"))
    label_valid = ds_label.GetRasterBand(1).ReadAsArray().astype(bool)
    ds_label = None
    
    # Intersect with boundary mask to get counts inside the administrative boundary
    feat_sum = int(feat_valid.sum())
    label_valid_sum = int((label_valid & feat_valid).sum())
    flood_sum = int((flood & feat_valid).sum())
    water_sum = int((water & feat_valid).sum())
    
    rows.append({
        "Wilayah": r.replace("_", " "),
        "Piksel Valid Batas": feat_sum,
        "Piksel Valid Label": label_valid_sum,
        "Piksel Banjir (UNOSAT)": flood_sum,
        "Piksel Air Permanen": water_sum
    })

df_label = pd.DataFrame(rows)
df_label

/usr/lib/python3.14/site-packages/osgeo/gdal.py:606: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


,Wilayah,Piksel Valid Batas,Piksel Valid Label,Piksel Banjir (UNOSAT),Piksel Air Permanen
0,Aceh Besar,28366694,28365443,1236843,647429
1,Aceh Tamiang,21636316,21635386,876070,1098925
2,Aceh Timur,53531470,53530030,2919804,2346567
3,Aceh Utara,26596884,26595884,4633938,3188065
4,Agam,22027974,21991761,264354,0
5,Banda Aceh,519063,519004,39245,17354
6,Bireuen,17831098,17830343,991351,632484
7,Langsa,2162309,2162100,104164,171408
8,Pasaman Barat,38201429,38199890,320599,0
9,Pidie,31591717,31589594,1620412,760528


## 2. Statistik Distribusi Tile Hasil Pemotongan (Tiling) dan Pembagian Dataset Spasial

Pada bagian ini, kita akan memuat data ringkasan hasil tiling dari `dataset/preprocessing_summary.csv` untuk mempresentasikan sebaran jumlah tile (positif vs background) dan jumlah piksel banjir setelah tiling.

In [3]:
TILE_SUMMARY_PATH = ROOT / "dataset/preprocessing_summary.csv"
df_tiles = pd.read_csv(TILE_SUMMARY_PATH)

# Rapikan kolom dan format untuk tabel skripsi
df_tiles_formatted = df_tiles[[
    "region", "split", "tile_count", "positive_tile_count", "background_tile_count", "flood_pixels", "valid_pixels"
]].copy()

df_tiles_formatted.columns = [
    "Wilayah", "Group Split", "Total Tile", "Tile Positif", "Tile Background", "Piksel Banjir (Tile)", "Piksel Valid (Tile)"
]

df_tiles_formatted["Wilayah"] = df_tiles_formatted["Wilayah"].str.replace("_", " ")
df_tiles_formatted = df_tiles_formatted.sort_values(by="Wilayah").reset_index(drop=True)

print("Tabel Sebaran Tile per Wilayah:")
display(df_tiles_formatted)

# Tampilkan total keseluruhan
total_tiles = df_tiles["tile_count"].sum()
total_pos = df_tiles["positive_tile_count"].sum()
total_bg = df_tiles["background_tile_count"].sum()
print(f"\nTotal Tile: {total_tiles} (Positif: {total_pos}, Background: {total_bg})")

# Tampilkan split CV vs Test
cv_tiles = df_tiles[df_tiles["split"] == "cv"]["tile_count"].sum()
cv_pos = df_tiles[df_tiles["split"] == "cv"]["positive_tile_count"].sum()
cv_bg = df_tiles[df_tiles["split"] == "cv"]["background_tile_count"].sum()

test_tiles = df_tiles[df_tiles["split"] == "test"]["tile_count"].sum()
test_pos = df_tiles[df_tiles["split"] == "test"]["positive_tile_count"].sum()
test_bg = df_tiles[df_tiles["split"] == "test"]["background_tile_count"].sum()

print(f"Cross-validation (10 wilayah): {cv_tiles} tile (Positif: {cv_pos}, Background: {cv_bg})")
print(f"Final Test Holdout (Aceh Utara): {test_tiles} tile (Positif: {test_pos}, Background: {test_bg})")

Tabel Sebaran Tile per Wilayah:


,Wilayah,Group Split,Total Tile,Tile Positif,Tile Background,Piksel Banjir (Tile),Piksel Valid (Tile)
0,Aceh Besar,cv,518,315,203,5429056,106956513
1,Aceh Tamiang,cv,465,328,137,4578520,82307919
2,Aceh Timur,cv,1063,543,520,26605176,206802875
3,Aceh Utara,test,493,332,161,22122708,102158650
4,Agam,cv,278,139,139,1257081,54576695
5,Banda Aceh,cv,16,16,0,326952,2532433
6,Bireuen,cv,303,186,117,3827030,67149979
7,Langsa,cv,41,37,4,933776,7395290
8,Pasaman Barat,cv,522,261,261,1462154,115359798
9,Pidie,cv,572,286,286,9351866,110864777



Total Tile: 4423 (Positif: 2526, Background: 1897)
Cross-validation (10 wilayah): 3930 tile (Positif: 2194, Background: 1736)
Final Test Holdout (Aceh Utara): 493 tile (Positif: 332, Background: 161)


## 3. Verifikasi Struktur Tensor Data Tile (.npz)

Untuk memastikan struktur data tile sudah sesuai dengan standar masukan model (7-channel dan multi-mask), mari kita muat satu sampel file `.npz` secara acak dan periksa struktur key serta dimensinya.

In [4]:
tile_dir = ROOT / "dataset/tiles/7ch/by_region/Banda_Aceh"
sample_tile_path = list(tile_dir.glob("*.npz"))[0]
data = np.load(sample_tile_path, allow_pickle=True)

print(f"Sampel file tile: {sample_tile_path.name}")
print("\nDaftar Key di dalam file .npz:")
for k in data.files:
    val = data[k]
    if isinstance(val, np.ndarray):
        print(f"- {k}: tipe={val.dtype}, shape={val.shape}")
    else:
        print(f"- {k}: tipe={type(val)}, value={val}")

# Verifikasi range nilai fitur
x = data["x"]
print(f"\nRange nilai Fitur (X): [{x.min():.4f}, {x.max():.4f}]")
for idx, ch in enumerate(data["channels"]):
    ch_slice = x[idx]
    print(f"  * Channel {idx+1} ({ch}): min={ch_slice.min():.4f}, max={ch_slice.max():.4f}")

Sampel file tile: Banda_Aceh_r000000_c000000.npz

Daftar Key di dalam file .npz:
- x: tipe=float32, shape=(7, 512, 512)
- y: tipe=uint8, shape=(1, 512, 512)
- valid_mask: tipe=uint8, shape=(1, 512, 512)
- water_river_mask: tipe=uint8, shape=(1, 512, 512)
- feature_valid_mask: tipe=uint8, shape=(1, 512, 512)
- s2_valid_mask: tipe=uint8, shape=(1, 512, 512)
- region: tipe=<U10, shape=()
- row: tipe=int64, shape=()
- col: tipe=int64, shape=()
- channels: tipe=<U10, shape=(7,)

Range nilai Fitur (X): [0.0000, 1.0000]
  * Channel 1 (VV): min=0.0000, max=1.0000
  * Channel 2 (VH): min=0.0000, max=1.0000
  * Channel 3 (Hue): min=0.0000, max=0.9986
  * Channel 4 (Saturation): min=0.0000, max=1.0000
  * Channel 5 (Value): min=0.0000, max=0.5304
  * Channel 6 (Slope): min=0.0000, max=0.5892
  * Channel 7 (HAND): min=0.0000, max=0.3813
